# lcz_interp_map_plus — ML-based temperature interpolation

Tests the Random Forest / OLS interpolation workflow with urban morphological parameters.
Each section is independent after `map_path` and `df` are set in §§1-2.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

import polars as pl
import pandas as pd
import numpy as np
import rasterio
import warnings
warnings.filterwarnings('ignore')

from lcz_get_map import lcz_get_map
from lcz_interp_map_plus import (
    lcz_interp_map_plus, lcz_interp_eval_plus,
    LCZInterpResult, _extract_lcz_params_at_stations,
    _compute_ols_vif, _check_missing_rasters,
)
from lcz_plot_interp import lcz_plot_interp

## 1 — Download LCZ map (Berlin)

In [2]:
map_path = lcz_get_map(city="Berlin", isave_map=False, lang = "pt")
print("map_path:", map_path)

map_path: /Users/co2map/.lcz4r_cache/clipped_2ed1dcfe644b4c6b.tif


## 2 — Load station data

In [3]:
df = pd.read_csv("lcz_data_berlin.csv", parse_dates=["date"])
print(f"Rows: {len(df)}, Stations: {df['station'].nunique()}, Date range: {df['date'].min()} -> {df['date'].max()}")
df.head(3)

Rows: 386354, Stations: 23, Date range: 2019-01-01 00:00:00 -> 2020-11-30 23:00:00


,Unnamed: 0,date,station,airT,Latitude,Longitude
0,806404,2019-01-01,airportthf,8.50000,52.467500,13.402100
1,806405,2019-01-01,airporttxl,7.90000,52.564400,13.308800
2,806406,2019-01-01,albrecht,8.04725,52.444594,13.348607


## 3 — RF interpolation with uncertainty
The `uncertainty_path` contains per-pixel prediction std (from tree ensemble).

In [4]:
result = lcz_interp_map_plus(
    map_path, df, var="airT", station_id="station",
    ml_model="rf",
    isave=True,
    year=2019, month=7, day=1, hour=12,
    use_lcz_params=True,
)

if result:
    print(f"Output: {result.path}")
    print(f"Uncertainty: {result.uncertainty_path}")
    print(f"Stations: {result.n_stations}, Features: {result.n_features}, Groups: {len(result.groups)}")
    if result.feature_importance:
        g = list(result.feature_importance.keys())[0]
        imp = result.feature_importance[g]
        top5 = sorted(imp.items(), key=lambda x: x[1], reverse=True)[:5]
        print(f"Top-5 features ({g}): {top5}")
    # Read uncertainty stats
    with rasterio.open(result.uncertainty_path) as src:
        unc = src.read(1)
        print(f"Uncertainty: min={np.nanmin(unc):.3f}, max={np.nanmax(unc):.3f}, mean={np.nanmean(unc):.3f}")

10:54:31 - lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...
10:54:31 - lcz_get_ucp - INFO - ============================================================
10:54:31 - lcz_get_ucp - INFO - Urban Parameter Processor Initialized
10:54:31 - lcz_get_ucp - INFO - ============================================================
10:54:31 - lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676
10:54:31 - lcz_get_ucp - INFO - Target CRS: EPSG:4326
10:54:31 - lcz_get_ucp - INFO - Target Shape: (377, 750)
10:54:31 - lcz_get_ucp - INFO - Workers: 7
10:54:31 - lcz_get_ucp - INFO - Cache: lcz4r_cache
10:54:31 - lcz_get_ucp - INFO - ============================================================
10:54:31 - lcz_get_ucp - INFO - 
10:54:31 - lcz_get_ucp - INFO - Processing 11 variables
10:54:31 - lcz_get_ucp - INFO - ============================================================
Variables:   0%|          | 0/11 [00:00<?, ?var/s]10:54:31 - lcz_get_ucp - INFO -   ✓ Using cached: eleva

Output: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpb1_l846y.tif
Uncertainty: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpkizlydp8.tif
Stations: 23, Features: 15, Groups: 1
Top-5 features (2019-07-01 12:00:00): [('elevation', 0.24659606961007746), ('cglc', 0.18404038587812047), ('tree', 0.15063605507518374), ('pop', 0.1310617934758499), ('lcz_PSF_mean', 0.05780676410576337)]
Uncertainty: min=0.521, max=1.847, mean=1.185


## 4 — OLS (Multiple Linear Regression)

In [5]:
result_ols = lcz_interp_map_plus(
    map_path, df, var="airT", station_id="station",
    ml_model="ols",
    isave=True,
    year=2019, month=7, day=1, hour=12,
)

if result_ols:
    print(f"Stations: {result_ols.n_stations}, Features: {result_ols.n_features}")
    if result_ols.cv_scores:
        for g, s in result_ols.cv_scores.items():
            print(f"  {g}: R2={s.get('r2',0):.3f}, residual SE={s.get('residual_se',0):.3f}")
    if result_ols.feature_importance:
        g = list(result_ols.feature_importance.keys())[0]
        coefs = result_ols.feature_importance[g]
        print(f"Coefficients ({g}):", {k: round(v, 4) for k, v in list(coefs.items())[:6]})
    # Uncertainty for OLS = residual SE (constant across grid)
    with rasterio.open(result_ols.uncertainty_path) as src:
        unc = src.read(1)
        print(f"OLS uncertainty (residual SE): {np.nanmean(unc):.3f}")

10:54:39 - lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...
10:54:39 - lcz_get_ucp - INFO - ============================================================
10:54:39 - lcz_get_ucp - INFO - Urban Parameter Processor Initialized
10:54:39 - lcz_get_ucp - INFO - ============================================================
10:54:39 - lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676
10:54:39 - lcz_get_ucp - INFO - Target CRS: EPSG:4326
10:54:39 - lcz_get_ucp - INFO - Target Shape: (377, 750)
10:54:39 - lcz_get_ucp - INFO - Workers: 7
10:54:39 - lcz_get_ucp - INFO - Cache: lcz4r_cache
10:54:39 - lcz_get_ucp - INFO - ============================================================
10:54:39 - lcz_get_ucp - INFO - 
10:54:39 - lcz_get_ucp - INFO - Processing 11 variables
10:54:39 - lcz_get_ucp - INFO - ============================================================
Variables:   0%|          | 0/11 [00:00<?, ?var/s]10:54:39 - lcz_get_ucp - INFO -   ✓ Using cached: eleva

Stations: 23, Features: 4
  2019-07-01 12:00:00: R2=0.433, residual SE=0.838
Coefficients (2019-07-01 12:00:00): {'intercept': 29.4548, 'cglc': 0.0253, 'elevation': -0.0709, 'pop': -0.0118, 'tree': 0.0155}
OLS uncertainty (residual SE): 0.838


## 5 — RF with LCZ morphological parameters + bbox

In [6]:
result_bbox = lcz_interp_map_plus(
    map_path, df, var="airT", station_id="station",
    ml_model="rf",
    use_lcz_params=True,
    bbox=(13.2, 52.4, 13.5, 52.6),  # crop to central Berlin
    year=2019, month=7, day=1, hour=12,
)

if result_bbox:
    with rasterio.open(result_bbox.path) as src:
        print(f"Cropped grid: {src.height}x{src.width} pixels")
        print(f"Bounds: {src.bounds}")
    print(f"Features: {result_bbox.n_features}")

10:54:43 - lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...
10:54:43 - lcz_get_ucp - INFO - ============================================================
10:54:43 - lcz_get_ucp - INFO - Urban Parameter Processor Initialized
10:54:43 - lcz_get_ucp - INFO - ============================================================
10:54:43 - lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676
10:54:43 - lcz_get_ucp - INFO - Target CRS: EPSG:4326
10:54:43 - lcz_get_ucp - INFO - Target Shape: (377, 750)
10:54:43 - lcz_get_ucp - INFO - Workers: 7
10:54:43 - lcz_get_ucp - INFO - Cache: lcz4r_cache
10:54:43 - lcz_get_ucp - INFO - ============================================================
10:54:43 - lcz_get_ucp - INFO - 
10:54:43 - lcz_get_ucp - INFO - Processing 11 variables
10:54:43 - lcz_get_ucp - INFO - ============================================================
Variables:   0%|          | 0/11 [00:00<?, ?var/s]10:54:43 - lcz_get_ucp - INFO -   ✓ Using cached: eleva

Cropped grid: 218x209 pixels
Bounds: BoundingBox(left=377538.5589360384, bottom=5807031.862960503, right=398438.5589360384, top=5828831.862960503)
Features: 23


## 6 — Visualize interpolated map

In [7]:
if result:
    fig = lcz_plot_interp(result.path, palette="muted", isave=True)
    fig.show()

10:54:46 - lcz_plot_interp - INFO - Output saved to {path /Users/co2map/Documents/lcz4r_python/LCZ4r_output/lcz_interp_map.html}


## 7 — Cross-validation (LOOCV) with cv_method column

In [8]:
cv_result = lcz_interp_eval_plus(
    map_path, df, var="airT", station_id="station",
    ml_model="rf",
    LOOCV=True,
    isave=True,
    year=2019, month=7, day=1, hour=12,
)

if not cv_result.is_empty():
    print(f"CV results: {len(cv_result)} predictions")
    print(f"Columns: {cv_result.columns}")
    print(f"\nPer-group metrics:")
    summary = cv_result.group_by("group").agg([
        pl.col("rmse").first().alias("rmse"),
        pl.col("mae").first().alias("mae"),
        pl.col("r2").first().alias("r2"),
        pl.len().alias("n_stations"),
    ])
    print(summary)
    rmse = np.sqrt(cv_result['residual'].pow(2).mean())
    mae = cv_result['residual'].abs().mean()
    print(f"\nOverall RMSE: {rmse:.3f} C")
    print(f"Overall MAE:  {mae:.3f} C")

10:55:03 - lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...
10:55:03 - lcz_get_ucp - INFO - ============================================================
10:55:03 - lcz_get_ucp - INFO - Urban Parameter Processor Initialized
10:55:03 - lcz_get_ucp - INFO - ============================================================
10:55:03 - lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676
10:55:03 - lcz_get_ucp - INFO - Target CRS: EPSG:4326
10:55:03 - lcz_get_ucp - INFO - Target Shape: (377, 750)
10:55:03 - lcz_get_ucp - INFO - Workers: 7
10:55:03 - lcz_get_ucp - INFO - Cache: lcz4r_cache
10:55:03 - lcz_get_ucp - INFO - ============================================================
10:55:03 - lcz_get_ucp - INFO - 
10:55:03 - lcz_get_ucp - INFO - Processing 11 variables
10:55:03 - lcz_get_ucp - INFO - ============================================================
Variables:   0%|          | 0/11 [00:00<?, ?var/s]10:55:03 - lcz_get_ucp - INFO -   ✓ Using cached: eleva

CV results: 15 predictions
Columns: ['station', 'group', 'observed', 'predicted', 'residual', 'cv_method', 'rmse', 'mae', 'r2']

Per-group metrics:
shape: (1, 5)
┌─────────────────────┬──────────┬──────────┬───────────┬────────────┐
│ group               ┆ rmse     ┆ mae      ┆ r2        ┆ n_stations │
│ ---                 ┆ ---      ┆ ---      ┆ ---       ┆ ---        │
│ str                 ┆ f64      ┆ f64      ┆ f64       ┆ u64        │
╞═════════════════════╪══════════╪══════════╪═══════════╪════════════╡
│ 2019-07-01 12:00:00 ┆ 1.360148 ┆ 1.088938 ┆ -0.495651 ┆ 15         │
└─────────────────────┴──────────┴──────────┴───────────┴────────────┘

Overall RMSE: 1.360 C
Overall MAE:  1.089 C


## 8 — Cross-validation (hold-out 80/20)

In [9]:
cv_hold = lcz_interp_eval_plus(
    map_path, df, var="airT", station_id="station",
    ml_model="rf",
    LOOCV=False,
    split_ratio=0.8,
    isave=True,
    year=2019, month=7, day=1, hour=12,
)

if not cv_hold.is_empty():
    print(f"Hold-out results: {len(cv_hold)} test predictions")
    rmse = np.sqrt(cv_hold['residual'].pow(2).mean())
    print(f"Overall RMSE: {rmse:.3f} C")
    print(f"Overall MAE:  {cv_hold['residual'].abs().mean():.3f} C")
    print(f"Overall R2:   {cv_hold['r2'].mean():.3f}")

10:55:07 - lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...
10:55:07 - lcz_get_ucp - INFO - ============================================================
10:55:07 - lcz_get_ucp - INFO - Urban Parameter Processor Initialized
10:55:07 - lcz_get_ucp - INFO - ============================================================
10:55:07 - lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676
10:55:07 - lcz_get_ucp - INFO - Target CRS: EPSG:4326
10:55:07 - lcz_get_ucp - INFO - Target Shape: (377, 750)
10:55:07 - lcz_get_ucp - INFO - Workers: 7
10:55:07 - lcz_get_ucp - INFO - Cache: lcz4r_cache
10:55:07 - lcz_get_ucp - INFO - ============================================================
10:55:07 - lcz_get_ucp - INFO - 
10:55:07 - lcz_get_ucp - INFO - Processing 11 variables
10:55:07 - lcz_get_ucp - INFO - ============================================================
Variables:   0%|          | 0/11 [00:00<?, ?var/s]10:55:07 - lcz_get_ucp - INFO -   ✓ Using cached: eleva

Hold-out results: 3 test predictions
Overall RMSE: 1.977 C
Overall MAE:  1.576 C
Overall R2:   -0.049


## 9 — LCZ parameter extraction helper

In [10]:
classes = np.array([1, 3, 5, 10, 12, 17])
feats, names = _extract_lcz_params_at_stations(classes)
print(f"Classes: {classes}")
print(f"Parameter names: {names}")
print(f"Feature matrix shape: {feats.shape}")
print(f"\nParameters for class 1 (Compact highrise):")
for name, val in zip(names, feats[0]):
    print(f"  {name:12s} = {val:.2f}")

Classes: [ 1  3  5 10 12 17]
Parameter names: ['svf_mean', 'BSF_mean', 'ISF_mean', 'PSF_mean', 'TSF_mean', 'HRE_mean', 'TRC_mean', 'SAD_mean', 'SAL_mean', 'AH_mean', 'z0']
Feature matrix shape: (6, 11)

Parameters for class 1 (Compact highrise):
  svf_mean     = 0.30
  BSF_mean     = 50.00
  ISF_mean     = 50.00
  PSF_mean     = 5.00
  TSF_mean     = 0.00
  HRE_mean     = 26.00
  TRC_mean     = 8.00
  SAD_mean     = 1.65
  SAL_mean     = 0.15
  AH_mean      = 175.00
  z0           = 2.00


## 10 — Small sample test (3 stations)

In [11]:
stations_small = df["station"].unique()[:3].tolist()
df_small = df[df["station"].isin(stations_small)]
print(f"Small dataset: {len(df_small)} rows, {df_small['station'].nunique()} stations")

result_small = lcz_interp_map_plus(
    map_path, df_small, var="airT", station_id="station",
    ml_model="rf",
    n_estimators=50,
    min_samples=2,
    year=2019, month=7, day=1, hour=12,
)

if result_small:
    print(f"OK — {result_small.n_stations} stations, {result_small.n_features} features")
else:
    print("Returned None (expected with too few stations)")

10:55:13 - lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...
10:55:13 - lcz_get_ucp - INFO - ============================================================
10:55:13 - lcz_get_ucp - INFO - Urban Parameter Processor Initialized
10:55:13 - lcz_get_ucp - INFO - ============================================================
10:55:13 - lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676
10:55:13 - lcz_get_ucp - INFO - Target CRS: EPSG:4326
10:55:13 - lcz_get_ucp - INFO - Target Shape: (377, 750)
10:55:13 - lcz_get_ucp - INFO - Workers: 7
10:55:13 - lcz_get_ucp - INFO - Cache: lcz4r_cache
10:55:13 - lcz_get_ucp - INFO - ============================================================
10:55:13 - lcz_get_ucp - INFO - 
10:55:13 - lcz_get_ucp - INFO - Processing 11 variables
10:55:13 - lcz_get_ucp - INFO - ============================================================


Small dataset: 50394 rows, 3 stations


Variables:   0%|          | 0/11 [00:00<?, ?var/s]10:55:13 - lcz_get_ucp - INFO -   ✓ Using cached: elevation_13.088_52.338_13.761_52.676.tif
10:55:14 - lcz_get_ucp - INFO -   ✓ Using cached: frc_esa_13.088_52.338_13.761_52.676.tif
10:55:14 - lcz_get_ucp - INFO -   ✓ Using cached: hgt_13.088_52.338_13.761_52.676.tif
10:55:14 - lcz_get_ucp - INFO -   ✓ Using cached: lf_13.088_52.338_13.761_52.676.tif
10:55:14 - lcz_get_ucp - INFO -   ✓ Using cached: lc_13.088_52.338_13.761_52.676.tif
10:55:14 - lcz_get_ucp - INFO -   ✓ Using cached: lb_13.088_52.338_13.761_52.676.tif
10:55:14 - lcz_get_ucp - INFO -   ✓ Using cached: lp_13.088_52.338_13.761_52.676.tif
Variables: 100%|██████████| 11/11 [00:00<00:00, 18.69var/s, ✓ urban]   
10:55:14 - lcz_get_ucp - INFO - 
10:55:14 - lcz_get_ucp - INFO - Processing GHSL Data
10:55:14 - lcz_get_ucp - INFO - ============================================================
10:55:14 - lcz_get_ucp - INFO - Using cached GHSL tile index
10:55:14 - lcz_get_ucp - INFO 

OK — 3 stations, 4 features


## 11 — Missing raster diagnostics

In [12]:
# Simulate a raster with mostly zeros
test_rasters = {
    "good": np.random.rand(100, 100).astype(np.float32),
    "bad_zeros": np.zeros((100, 100), dtype=np.float32),
    "bad_nan": np.full((100, 100), np.nan, dtype=np.float32),
    "sparse": np.where(np.random.rand(100, 100) > 0.9, 1.0, 0.0).astype(np.float32),
}
bad = _check_missing_rasters(test_rasters, min_valid_fraction=0.5)
print(f"Bad rasters detected: {bad}")

10:55:17 - lcz_interp_map_plus - WARNING - Raster 'bad_zeros': only 0.0% valid pixels — excluding.
10:55:17 - lcz_interp_map_plus - WARNING - Raster 'sparse': only 9.6% valid pixels — excluding.


Bad rasters detected: ['bad_zeros', 'sparse']


## 12 — OLS VIF diagnostics

In [13]:
# VIF for the training features from the RF result (use same feature matrix)
if result and result.feature_importance:
    g = list(result.feature_importance.keys())[0]
    imp = result.feature_importance[g]
    feature_names = list(imp.keys())
    print(f"Feature count: {len(feature_names)}")
    print("Note: VIF requires OLS. Run with ml_model='ols' for full VIF output.")

Feature count: 15
Note: VIF requires OLS. Run with ml_model='ols' for full VIF output.
